In [ ]:
import numpy as np

# --- 1. Global Configuration Constants ---
N_ROWS, N_COLS = 10, 20
N_STATES = N_ROWS * N_COLS
N_ACTIONS = 4
START_STATE = 0
GOAL_STATE = N_STATES - 1

REWARD_GOAL = 10.0
REWARD_BOMB = -5.0
REWARD_STEP = -0.1

EPISODES = 5000
ALPHA = 0.1
GAMMA = 0.9
EPSILON = 0.15

ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 0: Up, 1: Down, 2: Left, 3: Right

# --- 2. Environment Setup Function ---
def build_environment():
    walls = set()
    bombs = set()

    # Main vertical wall in the middle
    for r in range(2, 9):
        walls.add(r * N_COLS + 10)

    for r in range(0, 4):
        walls.add(r * N_COLS + 3)

    # Extra horizontal walls to create a maze-like structure
    for c in range(3, 8):
        walls.add(3 * N_COLS + c)
    for c in range(12, 18):
        walls.add(6 * N_COLS + c)

    # Bombs (Terminal penalty states)
    bombs.update([
        8 * N_COLS + 5, 8 * N_COLS + 6,
        1 * N_COLS + 15, 2 * N_COLS + 15,
        4 * N_COLS + 4, 4 * N_COLS + 5,
        7 * N_COLS + 14, 7 * N_COLS + 15,
        3 * N_COLS + 1, 9 * N_COLS + 9
    ])
    return walls, list(bombs)

WALL_STATES, BOMB_STATES = build_environment()

def step(s, a):
    # Executes action a in state s, returning (s_next, reward, done)
    r, c = divmod(s, N_COLS)
    dr, dc = ACTIONS[a]
    nr, nc = max(0, min(N_ROWS - 1, r + dr)), max(0, min(N_COLS - 1, c + dc))
    s_next = nr * N_COLS + nc

    # Bouncing off walls
    if s_next in WALL_STATES:
        s_next = s

    if s_next == GOAL_STATE:
        return s_next, REWARD_GOAL, True
    elif s_next in BOMB_STATES:
        return s_next, REWARD_BOMB, True

    # Step deduction to encourage shortest path
    return s_next, REWARD_STEP, False

# --- 3. Q-Learning Training Algorithm ---
def train_q_learning():
    Q = np.zeros((N_STATES, N_ACTIONS))
    last_updates = []

    print(f"Training for {EPISODES} episodes...")
    print(f"{'Episode':<10} | {'Avg Reward':<12} | {'Avg Steps':<10} | {'Avg Abs TD Err':<15}")
    print("-" * 55)

    rewards_log, steps_log, td_log = [], [], []

    # Algorithm:
    # 1. Initialize Q(s, a) arbitrarily (e.g., zeros) (Done above)
    # 2. Repeat (for each episode):
    #    a. Initialize state S
    #    b. Repeat (for each step of episode):
    #       i.   Choose A from S using policy derived from Q (epsilon-greedy)
    #       ii.  Take action A, observe R, S'
    #       iii. Q(S,A) = Q(S,A) + alpha * [R + gamma * max_a Q(S',a) - Q(S,A)]
    #       iv.  S = S'
    #    c. until S is terminal
    for episode in range(1, EPISODES + 1):
        s = START_STATE  # Reset agent to Start state
        ep_reward = 0
        ep_steps = 0
        ep_td_error = 0

        while s not in [GOAL_STATE] + BOMB_STATES:
            # Epsilon-greedy action selection
            if np.random.rand() < EPSILON:
                a = np.random.randint(N_ACTIONS)
            else:
                a = np.argmax(Q[s])

            s_next, reward, done = step(s, a)

            # Step 1: Find the best next Q-value
            best_next = 0.0 if done else np.max(Q[s_next])

            # Step 2: Calculate the TD Target
            td_target = reward + GAMMA * best_next

            # Step 3: Calculate the TD Error
            td_error  = td_target - Q[s, a]

            # Step 4: Update Q-Value
            if td_error != 0:
                Q[s, a] += ALPHA * td_error
                last_updates.append((episode, s, a, td_error, np.copy(Q[s])))
                if len(last_updates) > 25:
                    last_updates.pop(0)

            ep_reward += reward
            ep_steps += 1
            ep_td_error += abs(td_error)
            s = s_next

        rewards_log.append(ep_reward)
        steps_log.append(ep_steps)
        td_log.append(ep_td_error / ep_steps if ep_steps > 0 else 0)

        if episode % 500 == 0:
            avg_rew = np.mean(rewards_log[-500:])
            avg_step = np.mean(steps_log[-500:])
            avg_td = np.mean(td_log[-500:])
            print(f"{episode:<10} | {avg_rew:<12.2f} | {avg_step:<10.1f} | {avg_td:<15.4f}")

    return Q, last_updates

if __name__ == "__main__":
    np.random.seed(42)
    trained_Q, last_updates = train_q_learning()

    print("\nTop 25 Last Q-Table Updates:")
    print(f"{'Episode':<8} | {'State (r,c)':<12} | {'Action':<6} | {'TD Err':<8} | {'Q(Up)':<8} | {'Q(Down)':<8} | {'Q(Left)':<8} | {'Q(Right)':<8}")
    print("-" * 92)
    action_symbols = ['^', 'v', '<', '>']

    for ep, s, a, td, q_row in last_updates:
        r, c = divmod(s, N_COLS)
        state_str = f"{s}({r},{c})"
        q_str = " | ".join([f"{val:<8.4f}" for val in q_row])
        print(f"{ep:<8} | {state_str:<12} | {action_symbols[a]:<6} | {td:<8.4f} | {q_str}")

    print("\nOptimal Policy Grid (0:^, 1:v, 2:<, 3:>, #: Wall, X: Bomb, G: Goal):")
    policy_grid = np.empty((N_ROWS, N_COLS), dtype=str)

    for s in range(N_STATES):
        r, c = divmod(s, N_COLS)
        if s == GOAL_STATE:
            policy_grid[r, c] = 'G'
        elif s in BOMB_STATES:
            policy_grid[r, c] = 'X'
        elif s in WALL_STATES:
            policy_grid[r, c] = '#'
        else:
            policy_grid[r, c] = action_symbols[np.argmax(trained_Q[s])]

    for row in policy_grid:
        print(" ".join(row))

    print("\nTest Episode (Greedy Execution using Trained Q-Table):")
    s = START_STATE
    test_path = [s]
    test_reward = 0
    while s not in [GOAL_STATE] + BOMB_STATES:
        a = np.argmax(trained_Q[s])
        s_next, reward, done = step(s, a)
        test_reward += reward
        test_path.append(s_next)
        s = s_next
        if len(test_path) > N_STATES: # fail-safe
            break

    print(f"Total Reward: {test_reward:.1f}, Steps: {len(test_path)-1}")

    path_grid = np.empty((N_ROWS, N_COLS), dtype=str)
    for s in range(N_STATES):
        r, c = divmod(s, N_COLS)
        if s == GOAL_STATE:
            path_grid[r, c] = 'G'
        elif s == START_STATE:
            path_grid[r, c] = 'S'
        elif s in BOMB_STATES:
            path_grid[r, c] = 'X'
        elif s in WALL_STATES:
            path_grid[r, c] = '#'
        else:
            path_grid[r, c] = '.'

    # Mark the path
    for s in test_path:
        if s not in [START_STATE, GOAL_STATE]:
            r, c = divmod(s, N_COLS)
            path_grid[r, c] = '*'

    for row in path_grid:
        print(" ".join(row))


Training for 5000 episodes...
Episode    | Avg Reward   | Avg Steps  | Avg Abs TD Err 
-------------------------------------------------------
500        | -11.49       | 110.3      | 0.1057         
1000       | 1.73         | 49.2       | 0.0968         
1500       | 3.62         | 36.9       | 0.0397         
2000       | 3.52         | 36.4       | 0.0170         
2500       | 3.54         | 36.5       | 0.0101         
3000       | 3.41         | 35.4       | 0.0088         
3500       | 3.72         | 36.5       | 0.0075         
4000       | 3.54         | 36.5       | 0.0065         
4500       | 3.56         | 36.9       | 0.0052         
5000       | 3.59         | 36.6       | 0.0073         

Top 25 Last Q-Table Updates:
Episode  | State (r,c)  | Action | TD Err   | Q(Up)    | Q(Down)  | Q(Left)  | Q(Right)
--------------------------------------------------------------------------------------------
5000     | 31(1,11)     | v      | 0.0000   | 0.8345   | 1.2648   | 0.8345  

In [ ]:
import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt

# ======================================================================
# GLOBAL CONSTANTS
# ======================================================================

# --- Canvas ---
CANVAS_W         = 600
CANVAS_H         = 140
GROUND_Y         = 120

# --- Dino body ---
DINO_X           = 50
DINO_W           = 24
DINO_H           = 32
DINO_GROUND_Y    = 88.0

# --- Physics ---
GRAVITY          = 0.4
SHORT_JUMP_VY    = -7.0    # short hop arc (action 1)
LONG_JUMP_VY     = -9.0    # longer arc, clears wider obstacles (action 2)

# --- Game speed ---
CACTUS_SPEED     = 2.0     # px/frame: world scroll speed; every shared constant feeds both Python & JS

# --- Spawning margins ---
SPAWN_MARGIN     = 4       # px: objects spawn just past the right edge
OFFSCREEN_MARGIN = 4       # px: objects removed once fully past the left edge (mirrors SPAWN_MARGIN)

# --- Obstacles ---
OBS_TYPES        = {0: (16, 24), 1: (24, 24)}
CACTUS_GAP_MIN   = 100     # tighter gaps -> more frequent decisions
CACTUS_GAP_MAX   = 200

# --- Coin grid ---
COIN_GRID_ROWS    = 6       # coin detection rows down the canvas height
COIN_GRID_COLS    = 3       # coin detection columns along the lookahead
COIN_ROW_HEIGHT   = GROUND_Y // COIN_GRID_ROWS
COIN_SIZE         = 12
COIN_GAP_MIN      = 10
COIN_GAP_MAX      = 20

# --- State space: uniform distance bins ---
OBS_GRID_MAX     = 100                     # px: obstacle binning range
COIN_GRID_MAX    = 60                      # px: coin grid lookahead range
GRID_CELL_W      = COIN_GRID_MAX // COIN_GRID_COLS   # px: coin grid column width

# --- State & action space sizes ---
NUM_OBS_TYPES    = 2       # single vs double cactus
NUM_DIST_BINS    = 10      # equal bins of OBS_BIN_W px each across 0..OBS_GRID_MAX
OBS_BIN_W        = OBS_GRID_MAX // NUM_DIST_BINS   # px per bin (derived)
NUM_AIRBORNE     = 2       # 0=grounded, 1=in-air
NUM_STATES       = NUM_OBS_TYPES * NUM_DIST_BINS * NUM_AIRBORNE
NUM_ACTIONS      = 3

def rand_coin_y():
    return random.randint(0, GROUND_Y - COIN_SIZE - 1)

# --- Rewards ---
REWARD_SURVIVAL      =    1.0   # per step alive on ground
REWARD_AIRBORNE      =    0.0   # per step alive while airborne (much less than ground)
REWARD_CACTUS_CLEAR  =    8.0   # bonus for clearing; survival is the primary objective
REWARD_COIN_COLLECT  =    2.0   # extra coins along the survival trajectory
REWARD_CRASH         = -200.0

# --- Training hyperparameters ---
TRAIN_EPISODES  = 2500    # episodes of Q-learning to run
MAX_STEPS       = 12000   # step cap per episode (must exceed the 10k survival target)
EVAL_STEPS      = 15000   # greedy eval horizon (> 10,000)
LOG_EVERY       = 100     # print a training row every N episodes
ALPHA           = 0.10    # learning rate: how much new info overwrites old Q-values
GAMMA           = 0.99    # discount factor: weight of future rewards vs immediate reward
INIT_Q          = 0.0     # starting Q-value for every (state, action) pair
EPSILON_START   = 0.15    # explore jump timing early
EPSILON_END     = 0.00    # fully greedy by the end of training
SEED            = 42      # random seed for reproducible runs


# ======================================================================
# 1. GOOGLE COLAB INTERACTIVE HTML5 CANVAS (TOP-LEVEL VISUALIZER)
# ======================================================================

def render_colab_dino(agent=None):
    if agent is not None and hasattr(agent, "q_table"):
        q_json = json.dumps(agent.q_table.tolist())
    elif isinstance(agent, (list, np.ndarray)):
        q_json = json.dumps(np.array(agent).tolist())
    else:
        q_json = "null"

    html_markup = f"""
    <div style="font-family: monospace; text-align: center; background: #0F172A; padding: 24px; border-radius: 18px; color: white; max-width: 1440px; margin: 0 auto; box-shadow: 0 16px 40px rgba(0,0,0,0.6);">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px; padding: 0 8px;">
        <h3 style="margin: 0; color: #38BDF8; font-size: 22px; font-weight: bold;">Chrome Dino RL - Tabular Q-Learning ({NUM_ACTIONS} Actions, {NUM_STATES} States)</h3>
        <button id="toggleBtn" onclick="togglePlayMode()" style="background: #10B981; color: white; border: none; padding: 8px 20px; border-radius: 24px; font-size: 15px; font-weight: bold; cursor: pointer;">
          Mode: AI Autoplay (ON)
        </button>
      </div>

      <canvas id="dinoCanvas" width="1200" height="260" style="width: 100%; max-width: 1200px; height: auto; background: #FFFFFF; border-radius: 12px; box-shadow: 0 6px 20px rgba(0,0,0,0.35); cursor: pointer; display: block; margin: 0 auto;"></canvas>

      <div style="margin-top: 16px; background: #1E293B; border-radius: 12px; padding: 16px; border: 1px solid #334155; text-align: left;">
        <div style="display: flex; justify-content: space-between; font-size: 16px; color: #94A3B8; margin-bottom: 10px;">
          <span style="font-weight: bold; color: #38BDF8;">[STATE PANEL] AGENT PERCEPTION & KINEMATICS</span>
          <span id="stateIndexText" style="color: #FCD34D; font-weight: bold; font-size: 18px;">State Index: --</span>
        </div>
        <canvas id="stateCanvas" width="1400" height="300" style="width: 100%; max-width: 1400px; height: auto; background: #0F172A; border-radius: 8px; display: block;"></canvas>
      </div>

      <div style="display: flex; justify-content: space-around; margin-top: 14px; font-size: 18px; background: #1E293B; padding: 12px 20px; border-radius: 10px; border: 1px solid #334155;">
        <span>Coins: <b id="dinoCoins" style="color: #FBBF24;">0</b></span>
        <span>Cactuses: <b id="dinoCacti" style="color: #F43F5E;">0</b></span>
        <span>Action: <b id="actionBadge" style="color: #38BDF8;">RUN (0)</b></span>
      </div>

      <div id="hintText" style="margin-top: 14px; font-size: 15px; color: #94A3B8;">
        AI Autoplay Active: Press 'A' or click button to toggle Manual Play.
      </div>
    </div>

    <script>
    (function() {{
      const C = {{
        CANVAS_W       : {CANVAS_W},
        GROUND_Y       : {GROUND_Y},
        DINO_X         : {DINO_X},
        DINO_W         : {DINO_W},
        DINO_H         : {DINO_H},
        DINO_GROUND_Y  : {DINO_GROUND_Y},
        GRAVITY        : {GRAVITY},
        SHORT_JUMP_VY  : {SHORT_JUMP_VY},
        LONG_JUMP_VY   : {LONG_JUMP_VY},
        CACTUS_SPEED   : {CACTUS_SPEED},
        CACTUS_W       : {[OBS_TYPES[0][0], OBS_TYPES[1][0]]},
        CACTUS_H       : {OBS_TYPES[0][1]},
        CACTUS_GAP_MIN : {CACTUS_GAP_MIN},
        CACTUS_GAP_MAX : {CACTUS_GAP_MAX},
        COIN_SIZE      : {COIN_SIZE},
        COIN_GAP_MIN   : {COIN_GAP_MIN},
        COIN_GAP_MAX   : {COIN_GAP_MAX},
        COIN_GRID_MAX  : {COIN_GRID_MAX},
        COIN_GRID_ROWS : {COIN_GRID_ROWS},
        COIN_GRID_COLS : {COIN_GRID_COLS},
        OBS_GRID_MAX   : {OBS_GRID_MAX},
        OBS_BIN_W      : {OBS_BIN_W},
        NUM_DIST_BINS  : {NUM_DIST_BINS},
        NUM_AIRBORNE   : {NUM_AIRBORNE},
        GRID_CELL_W     : {GRID_CELL_W},
        COIN_ROW_H     : {COIN_ROW_HEIGHT},
        SPAWN_MARGIN   : {SPAWN_MARGIN},
        OFFSCREEN_MARGIN : {OFFSCREEN_MARGIN},
      }};

      function randCoinY() {{
        return Math.floor(Math.random() * (C.GROUND_Y - C.COIN_SIZE));
      }}

      const qTable = {q_json};
      const canvas = document.getElementById("dinoCanvas");
      const ctx = canvas.getContext("2d");
      const sCanvas = document.getElementById("stateCanvas");
      const sCtx = sCanvas.getContext("2d");
      const toggleBtn = document.getElementById("toggleBtn");
      const hintEl = document.getElementById("hintText");

      let autoPlay = (qTable !== null);
      if (!autoPlay) {{
        toggleBtn.innerText = "Mode: Manual Play (No Q-Table)";
        toggleBtn.style.background = "#8B5CF6";
        hintEl.innerHTML = "No Q-Table: Manual Play active — 'S'/Left Click = Short Jump | 'D'/Right Click = Long Jump.";
      }}

      let dino = {{ x: C.DINO_X, y: C.DINO_GROUND_Y, w: C.DINO_W, h: C.DINO_H, vy: 0, isJumping: false }};

      let cacti = [];
      let coins = [];
      let cactusDist = 0, cactusGap = Math.floor(Math.random()*(C.CACTUS_GAP_MAX - C.CACTUS_GAP_MIN + 1) + C.CACTUS_GAP_MIN);
      let coinDist = 0, coinGap = Math.floor(Math.random()*(C.COIN_GAP_MAX - C.COIN_GAP_MIN + 1) + C.COIN_GAP_MIN);

      let coinsCount = 0, cactiCount = 0, tick = 0;
      let manualAction = 0;
      let crashed = false;

      window.togglePlayMode = function() {{
        if (qTable === null) {{
          alert("No trained Q-Table! Run train_dino_agent() first.");
          return;
        }}
        autoPlay = !autoPlay;
        if (autoPlay) {{
          toggleBtn.innerText = "Mode: AI Autoplay (ON)";
          toggleBtn.style.background = "#10B981";
          hintEl.innerHTML = "AI Autoplay Active: Press 'A' to toggle Manual Play. After a crash, press S/D or click to restart.";
          crashed = false;
        }} else {{
          toggleBtn.innerText = "Mode: Manual Play (OFF)";
          toggleBtn.style.background = "#8B5CF6";
          hintEl.innerHTML = "Manual Play: 'S'/Left Click = Short Jump | 'D'/Right Click = Long Jump";
        }}
      }};

      window.addEventListener("keydown", function(e) {{
        if (e.code === "KeyS") {{
          e.preventDefault();
          if (crashed) {{ resetGame(); }} else if (!autoPlay) {{ manualAction = 1; }}
        }} else if (e.code === "KeyD") {{
          e.preventDefault();
          if (crashed) {{ resetGame(); }} else if (!autoPlay) {{ manualAction = 2; }}
        }} else if (e.code === "KeyA") {{
          window.togglePlayMode();
        }}
      }});

      canvas.addEventListener("mousedown", function(e) {{
        if (crashed) {{ resetGame(); return; }}
        if (!autoPlay) {{
          manualAction = (e.button === 0) ? 1 : (e.button === 2 ? 2 : 0);
        }}
      }});
      canvas.addEventListener("contextmenu", e => e.preventDefault());

      function resetGame() {{
        crashed = false; coinsCount = 0; cactiCount = 0; manualAction = 0;
        dino.y = C.DINO_GROUND_Y; dino.vy = 0; dino.isJumping = false;
        cacti = []; coins = [];
        cactusDist = 0; cactusGap = Math.floor(Math.random()*(C.CACTUS_GAP_MAX - C.CACTUS_GAP_MIN + 1) + C.CACTUS_GAP_MIN);
        coinDist = 0; coinGap = Math.floor(Math.random()*(C.COIN_GAP_MAX - C.COIN_GAP_MIN + 1) + C.COIN_GAP_MIN);
      }}

      function getNearestCactus() {{
        let a = cacti.filter(c => c.x + c.w >= dino.x);
        return a.length ? a.reduce((m, c) => c.x < m.x ? c : m) : null;
      }}

      function getNearestCoin() {{
        let a = coins.filter(c => !c.collected && c.x + C.COIN_SIZE >= C.DINO_X);
        return a.length ? a.reduce((m, c) => c.x < m.x ? c : m) : null;
      }}

      function getDiscreteState() {{
        let frontX = dino.x + dino.w;

        let c    = getNearestCactus();
        let dist = c ? Math.floor(c.x - frontX) : 999;
        let obsType = c ? c.type : 0;

        // NUM_DIST_BINS equal bins of OBS_BIN_W px each across 0..OBS_GRID_MAX
        let distBin = Math.min(C.NUM_DIST_BINS - 1, Math.floor(Math.max(0, dist) / (C.OBS_GRID_MAX / C.NUM_DIST_BINS)));

        let coinBlock = 0;
        let coin = getNearestCoin();
        if (coin) {{
          let cd = Math.max(0, Math.floor(coin.x - frontX));
          if (cd < C.COIN_GRID_MAX) {{
            let col = Math.min(C.COIN_GRID_COLS - 1, Math.floor(cd / C.GRID_CELL_W));
            let row = Math.min(C.COIN_GRID_ROWS - 1, Math.floor(coin.y / C.COIN_ROW_H));
            coinBlock = row * C.COIN_GRID_COLS + col + 1;
          }}
        }}

        let airborne = dino.isJumping ? 1 : 0;
        let s = (obsType * C.NUM_DIST_BINS * C.NUM_AIRBORNE) + (distBin * C.NUM_AIRBORNE) + airborne;
        return {{ s, obsType, distBin, coinBlock, airborne, dist }};
      }}

      function drawPixelDino(ctx, x, y, isJumping, isCrash, t) {{
        ctx.fillStyle = isCrash ? "#DC2626" : "#1E293B";
        ctx.fillRect(x+4, y+8, 16, 16); ctx.fillRect(x+10, y, 14, 10);
        ctx.fillStyle = isCrash ? "#FEE2E2" : "#FFFFFF"; ctx.fillRect(x+18, y+2, 2, 2);
        ctx.fillStyle = isCrash ? "#DC2626" : "#1E293B"; ctx.fillRect(x+18, y+14, 4, 3);
        if (isJumping || isCrash) {{
          ctx.fillRect(x+6, y+24, 3, 8); ctx.fillRect(x+13, y+22, 3, 6);
        }} else {{
          let leg = (Math.floor(t/5) % 2 === 0) ? 0 : 2;
          ctx.fillRect(x+6, y+24-leg, 3, 8); ctx.fillRect(x+13, y+24-(2-leg), 3, 8);
        }}
      }}

      function drawPixelCactus(ctx, x, y, type) {{
        ctx.fillStyle = "#15803D";
        if (type === 0) {{
          ctx.fillRect(x+5, y, 6, 24); ctx.fillRect(x+1, y+6, 4, 3);
          ctx.fillRect(x+1, y+6, 3, 8); ctx.fillRect(x+11, y+9, 4, 3); ctx.fillRect(x+12, y+9, 3, 7);
        }} else {{
          ctx.fillRect(x+4, y+2, 5, 22); ctx.fillRect(x+1, y+7, 3, 3);
          ctx.fillRect(x+1, y+7, 2, 7); ctx.fillRect(x+9, y+9, 3, 3); ctx.fillRect(x+10, y+9, 2, 6);
          ctx.fillRect(x+19, y, 6, 24); ctx.fillRect(x+15, y+6, 4, 3);
          ctx.fillRect(x+15, y+6, 3, 8); ctx.fillRect(x+25, y+8, 4, 3); ctx.fillRect(x+26, y+8, 3, 7);
        }}
      }}

      function drawPixelCoin(ctx, x, y) {{
        ctx.beginPath();
        ctx.arc(x + C.COIN_SIZE / 2, y + C.COIN_SIZE / 2, C.COIN_SIZE / 2, 0, Math.PI * 2);
        ctx.fillStyle = "#F59E0B";
        ctx.fill();
      }}

      function drawStatePanel(sCtx, st, action, q_r, q_s, q_l) {{
        sCtx.fillStyle = "#0F172A";
        sCtx.fillRect(0, 0, 1400, 300);
        sCtx.setLineDash([]);

        const DIM = "#334155", MID = "#475569", TITLE = "#38BDF8";
        const binDesc = st.distBin < C.NUM_DIST_BINS ? (st.distBin*C.OBS_BIN_W) + "-" + (st.distBin*C.OBS_BIN_W + C.OBS_BIN_W - 1) + "px" : "--";

        function title(txt, x) {{
          sCtx.font = "bold 20px Arial, sans-serif"; sCtx.fillStyle = TITLE;
          sCtx.textAlign = "left"; sCtx.fillText(txt, x, 30);
        }}

        // -- Section 1: Coin Grid ----------------------------------
        title("COIN DETECTION", 12);

        const bW = 70, bH = 34, gx = 8, gy = 2, sx = 100, sy = 80;
        const colH = ["0-" + (C.GRID_CELL_W-1) + "px", C.GRID_CELL_W + "-" + (C.GRID_CELL_W*2-1) + "px", C.GRID_CELL_W*2 + "-" + (C.COIN_GRID_MAX-1) + "px"];
        const rowH = Array.from({{ length: C.COIN_GRID_ROWS }}, (_, r) => "y" + (r*C.COIN_ROW_H) + "-" + ((r+1)*C.COIN_ROW_H));

        sCtx.textAlign = "center"; sCtx.font = "18px Arial, sans-serif"; sCtx.fillStyle = MID;
        for (let c = 0; c < C.COIN_GRID_COLS; c++)
          sCtx.fillText(colH[c], sx + c * (bW + gx) + bW / 2, sy - 6);

        for (let r = 0; r < C.COIN_GRID_ROWS; r++) {{
          sCtx.textAlign = "right"; sCtx.font = "18px Arial, sans-serif"; sCtx.fillStyle = MID;
          sCtx.fillText(rowH[r], sx - 8, sy + r * (bH + gy) + bH / 2 + 4);
          for (let c = 0; c < C.COIN_GRID_COLS; c++) {{
            let cell = r * C.COIN_GRID_COLS + c + 1, bx = sx + c * (bW + gx), by = sy + r * (bH + gy);
            let on = (st.coinBlock === cell);
            sCtx.strokeStyle = on ? "#FCD34D" : DIM; sCtx.lineWidth = on ? 2 : 1;
            sCtx.strokeRect(bx, by, bW, bH);
            if (on) {{
              sCtx.textAlign = "center";
              sCtx.font = "bold 17px Arial, sans-serif"; sCtx.fillStyle = "#FCD34D";
              sCtx.fillText("COIN", bx + bW / 2, by + bH / 2 + 5);
            }}
          }}
        }}

        // -- Section 2: Obstacle Distance Bins ----------------------
        title("OBSTACLE DISTANCE", 375);

        const obW = 48, obH = 180, obGx = 6, obX = 375, syObs = 58;
        const obColors = ["#EF4444", "#F04F32", "#F97316", "#FA8C1A", "#F59E0B",
                          "#EAB308", "#84CC16", "#22C55E", "#10B981", "#14B8A6"];

        for (let i = 0; i < C.NUM_DIST_BINS; i++) {{
          let bx = obX + i * (obW + obGx), by = syObs, on = (st.distBin === i);
          sCtx.strokeStyle = on ? obColors[i] : DIM; sCtx.lineWidth = on ? 3 : 1;
          sCtx.strokeRect(bx, by, obW, obH);
          sCtx.textAlign = "center";
          sCtx.font = on ? "bold 15px Arial, sans-serif" : "13px Arial, sans-serif";
          sCtx.fillStyle = on ? obColors[i] : MID;
          sCtx.fillText((i*C.OBS_BIN_W) + "-" + (i*C.OBS_BIN_W + C.OBS_BIN_W - 1), bx + obW / 2, by + 25);
          if (on) {{
            sCtx.fillStyle = obColors[i] + "20";
            sCtx.fillRect(bx + 2, by + 2, obW - 4, obH - 4);
          }}
        }}

        // Airborne status + obstacle info
        let airY = syObs + obH + 12;
        let airColor = st.airborne ? "#8B5CF6" : "#0284C7";
        sCtx.fillStyle = airColor; sCtx.fillRect(obX, airY, 300, 30);
        sCtx.fillStyle = "#FFF"; sCtx.textAlign = "center";
        sCtx.font = "bold 14px Arial, sans-serif";
        sCtx.fillText(st.airborne ? "IN AIR (can't jump)" : "GROUNDED (can jump)", obX + 150, airY + 21);
        sCtx.textAlign = "left"; sCtx.font = "13px Arial, sans-serif"; sCtx.fillStyle = "#94A3B8";
        sCtx.fillText("Obstacle: " + (st.obsType === 0 ? "Single" : "Double") + " | bin " + st.distBin + " (" + binDesc + ")", obX + 310, airY + 21);

        // -- Section 3: Q-Values & Decision -------------------------
        title("Q-VALUES & DECISION", 955);

        const dx = 955;
        let aColor = action === 0 ? "#0284C7" : action === 1 ? "#10B981" : "#EF4444";
        let aText = action === 0 ? "RUN (0)" : action === 1 ? "SHORT JUMP (1)" : "LONG JUMP (2)";
        sCtx.fillStyle = aColor; sCtx.fillRect(dx, 58, 285, 44);
        sCtx.fillStyle = "#FFF"; sCtx.textAlign = "center";
        sCtx.font = "bold 18px Arial, sans-serif";
        sCtx.fillText(aText, dx + 142, 87);

        sCtx.textAlign = "left"; sCtx.font = "18px Arial, sans-serif"; sCtx.fillStyle = "#94A3B8";
        sCtx.fillText("Q(RUN):        " + (q_r !== null ? q_r.toFixed(1) : "--"), dx, 135);
        sCtx.fillText("Q(SHORT JUMP): " + (q_s !== null ? q_s.toFixed(1) : "--"), dx, 160);
        sCtx.fillText("Q(LONG JUMP):  " + (q_l !== null ? q_l.toFixed(1) : "--"), dx, 185);
      }}

      function gameLoop() {{
        tick++;

        let stPre = getDiscreteState();
        let s     = stPre.s;
        let hasQ  = qTable !== null && qTable[s] !== undefined;
        let q_r   = hasQ ? qTable[s][0] : null;
        let q_s   = hasQ ? qTable[s][1] : null;
        let q_l   = hasQ ? qTable[s][2] : null;

        let action = 0;
        if (autoPlay && hasQ) {{
          let best = Math.max(q_r, q_s, q_l);
          let cands = [];
          if (q_r === best) cands.push(0);
          if (q_s === best) cands.push(1);
          if (q_l === best) cands.push(2);
          action = cands[Math.floor(Math.random() * cands.length)];
        }} else if (manualAction > 0) {{
          action = manualAction; manualAction = 0;
        }}

        if (!crashed) {{
          if ((action === 1 || action === 2) && !dino.isJumping) {{
            dino.vy = action === 1 ? C.SHORT_JUMP_VY : C.LONG_JUMP_VY;
            dino.isJumping = true;
          }}
          if (dino.isJumping) {{
            dino.y += dino.vy; dino.vy += C.GRAVITY;
            if (dino.y >= C.DINO_GROUND_Y) {{ dino.y = C.DINO_GROUND_Y; dino.vy = 0; dino.isJumping = false; }}
          }}

          for (let i = 0; i < cacti.length; i++) {{
            let c = cacti[i];
            c.x -= C.CACTUS_SPEED;
            if (c.x + c.w < dino.x && !c.cleared) {{ c.cleared = true; cactiCount++; }}
            if (c.x + c.w <= -C.OFFSCREEN_MARGIN) {{ cacti.splice(i, 1); i--; }}
          }}

          cactusDist += C.CACTUS_SPEED;
          if (cactusDist >= cactusGap) {{
            cactusDist = 0;
            cactusGap = Math.floor(Math.random()*(C.CACTUS_GAP_MAX - C.CACTUS_GAP_MIN + 1) + C.CACTUS_GAP_MIN);
            let t = Math.random() < 0.5 ? 0 : 1;
            cacti.push({{ x: C.CANVAS_W + C.SPAWN_MARGIN, w: C.CACTUS_W[t], h: C.CACTUS_H, type: t, cleared: false }});
          }}

          for (let i = 0; i < coins.length; i++) {{
            let coin = coins[i];
            coin.x -= C.CACTUS_SPEED;
            if (!coin.collected) {{
              let hx = dino.x < coin.x + C.COIN_SIZE && dino.x + dino.w > coin.x;
              let hy = dino.y < coin.y + C.COIN_SIZE && dino.y + dino.h > coin.y;
              if (hx && hy) {{ coin.collected = true; coinsCount++; }}
            }}
            if ((coin.collected && coin.x + C.COIN_SIZE < dino.x) || coin.x + C.COIN_SIZE <= -C.OFFSCREEN_MARGIN) {{ coins.splice(i, 1); i--; }}
          }}

          coinDist += C.CACTUS_SPEED;
          if (coinDist >= coinGap) {{
            coinDist = 0;
            coinGap = Math.floor(Math.random()*(C.COIN_GAP_MAX - C.COIN_GAP_MIN + 1) + C.COIN_GAP_MIN);
            coins.push({{ x: C.CANVAS_W + C.SPAWN_MARGIN, y: randCoinY(), collected: false }});
          }}

          for (let i = 0; i < cacti.length; i++) {{
            let c = cacti[i];
            if (dino.x < c.x+c.w && dino.x+dino.w > c.x && dino.y+dino.h > C.GROUND_Y - c.h) {{
              crashed = true;
              break;
            }}
          }}
        }}

        let st = getDiscreteState();

        document.getElementById("dinoCoins").innerText = coinsCount;
        document.getElementById("dinoCacti").innerText = cactiCount;
        let ab = document.getElementById("actionBadge");
        ab.innerText = action === 0 ? "RUN (0)" : action === 1 ? "SHORT (1)" : "LONG (2)";
        ab.style.color = action === 0 ? "#38BDF8" : action === 1 ? "#10B981" : "#F43F5E";
        document.getElementById("stateIndexText").innerText = "State Index: " + st.s;

        ctx.save();
        ctx.scale(2, 2);
        ctx.fillStyle = crashed ? "#FEF2F2" : "#FFFFFF"; ctx.fillRect(0, 0, C.GROUND_Y * 5, C.GROUND_Y + 10);
        ctx.strokeStyle = "#64748B"; ctx.lineWidth = 2;
        ctx.beginPath(); ctx.moveTo(0, C.GROUND_Y); ctx.lineTo(C.CANVAS_W, C.GROUND_Y); ctx.stroke();
        ctx.fillStyle = "#94A3B8";
        for (let gx = 20; gx < 590; gx += 40) {{ ctx.fillRect(gx, C.GROUND_Y + 2, 8, 2); }}

        let fx = dino.x + dino.w;

        // Obstacle distance bins (NUM_DIST_BINS equal bins of OBS_BIN_W px)
        ctx.save();
        ctx.strokeStyle = "rgba(239,68,68,0.25)"; ctx.lineWidth = 0.6; ctx.setLineDash([3,3]);
        for (let i = 1; i < C.NUM_DIST_BINS; i++) {{ ctx.beginPath(); ctx.moveTo(fx + i*(C.OBS_GRID_MAX/C.NUM_DIST_BINS), 0); ctx.lineTo(fx + i*(C.OBS_GRID_MAX/C.NUM_DIST_BINS), C.GROUND_Y); ctx.stroke(); }}

        // Coin detection grid (COIN_GRID_COLS x COIN_GRID_ROWS)
        let gridX = Array.from({{ length: C.COIN_GRID_COLS + 1 }}, (_, c) => fx + c * C.GRID_CELL_W);
        let gridY = Array.from({{ length: C.COIN_GRID_ROWS + 1 }}, (_, r) => r * C.COIN_ROW_H);

        if (st.coinBlock > 0) {{
          let cb = st.coinBlock - 1;
          let cr = Math.floor(cb / C.COIN_GRID_COLS);
          let cc = cb % C.COIN_GRID_COLS;
          ctx.fillStyle = "rgba(245,158,11,0.22)";
          ctx.fillRect(gridX[cc], gridY[cr], C.GRID_CELL_W, C.COIN_ROW_H);
        }}

        ctx.strokeStyle = "rgba(100,116,139,0.25)"; ctx.lineWidth = 0.8; ctx.setLineDash([3,3]);
        for (let x of gridX) {{ ctx.beginPath(); ctx.moveTo(x, 0); ctx.lineTo(x, C.GROUND_Y); ctx.stroke(); }}
        for (let y of gridY) {{ ctx.beginPath(); ctx.moveTo(fx, y); ctx.lineTo(fx+C.COIN_GRID_MAX, y); ctx.stroke(); }}
        ctx.restore();

        drawPixelDino(ctx, dino.x, dino.y, dino.isJumping, crashed, tick);
        cacti.forEach(c => drawPixelCactus(ctx, c.x, C.GROUND_Y - c.h, c.type));
        coins.forEach(c => {{ if (!c.collected) drawPixelCoin(ctx, c.x, c.y); }});

        if (crashed) {{
          ctx.fillStyle = "rgba(220,38,38,0.85)"; ctx.fillRect(130,35,340,50);
          ctx.fillStyle = "#FFF"; ctx.font = "bold 13px monospace"; ctx.textAlign = "center";
          ctx.fillText("COLLISION! S/D or Click to Restart", 300, 65);
        }}

        ctx.restore();
        drawStatePanel(sCtx, st, action, q_r, q_s, q_l);
        requestAnimationFrame(gameLoop);
      }}
      requestAnimationFrame(gameLoop);
    }})();
    </script>
    """
    try:
        from IPython.display import HTML, display
        display(HTML(html_markup))
    except ImportError:
        pass
    out_path = "section/ch07_figures/colab_dino_canvas.html" if os.path.exists("section") else "colab_dino_canvas.html"
    if os.path.dirname(out_path):
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(html_markup)
    print(f"Saved HTML5 Canvas to: {out_path}")


# ======================================================================
# 2. CHROME DINOSAUR 2D PHYSICS SIMULATION ENVIRONMENT
# ======================================================================

class ChromeDinoEnv:
    def __init__(self, max_steps=MAX_STEPS):
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.dino_y      = DINO_GROUND_Y
        self.dino_vy     = 0.0
        self.is_jumping  = False
        self.step_count  = 0
        self.cacti_cleared = 0
        self.coins_collected = 0
        self.cacti = []
        self.coins = []
        self._cactus_dist = 0.0
        self._coin_dist   = 0.0
        self._cactus_gap  = random.randint(CACTUS_GAP_MIN, CACTUS_GAP_MAX)
        self._coin_gap    = random.randint(COIN_GAP_MIN, COIN_GAP_MAX)
        return self._get_state()

    def _nearest_cactus(self):
        active = [c for c in self.cacti if c['x'] + c['w'] >= DINO_X]
        return min(active, key=lambda c: c['x']) if active else None

    def _get_state(self):
        c         = self._nearest_cactus()
        dist_obs = c['x'] - (DINO_X + DINO_W) if c else 999.0
        obs_type = c['type'] if c else 0

        # NUM_DIST_BINS equal bins of OBS_BIN_W px each across 0..OBS_GRID_MAX
        dist_bin  = NUM_DIST_BINS - 1 if dist_obs >= OBS_GRID_MAX else int(max(0.0, dist_obs) // OBS_BIN_W)

        airborne = 1 if self.is_jumping else 0

        return obs_type * NUM_DIST_BINS * NUM_AIRBORNE + dist_bin * NUM_AIRBORNE + airborne

    def step(self, action):
        self.step_count += 1
        reward = 0.0
        done   = False

        # Case 1: survival tick (airborne is worth less than ground)
        if self.is_jumping:
            reward += REWARD_AIRBORNE
        else:
            reward += REWARD_SURVIVAL

        if action in (1, 2) and not self.is_jumping:
            self.dino_vy    = SHORT_JUMP_VY if action == 1 else LONG_JUMP_VY
            self.is_jumping = True

        if self.is_jumping:
            self.dino_y  += self.dino_vy
            self.dino_vy += GRAVITY
            if self.dino_y >= DINO_GROUND_Y:
                self.dino_y  = DINO_GROUND_Y
                self.dino_vy = 0.0
                self.is_jumping = False

        for c in self.cacti:
            c['x'] -= CACTUS_SPEED
            # Case 2: cactus cleared bonus
            if c['x'] + c['w'] < DINO_X and not c['cleared']:
                c['cleared'] = True
                self.cacti_cleared += 1
                reward += REWARD_CACTUS_CLEAR
        self.cacti = [c for c in self.cacti if c['x'] + c['w'] > -OFFSCREEN_MARGIN]

        self._cactus_dist += CACTUS_SPEED
        if self._cactus_dist >= self._cactus_gap:
            self._cactus_dist = 0.0
            self._cactus_gap  = random.randint(CACTUS_GAP_MIN, CACTUS_GAP_MAX)
            ctype = random.choice([0, 1])
            self.cacti.append({'x': CANVAS_W + SPAWN_MARGIN, 'type': ctype,
                               'w': OBS_TYPES[ctype][0], 'h': OBS_TYPES[ctype][1],
                               'cleared': False})

        for coin in self.coins:
            coin['x'] -= CACTUS_SPEED
            if not coin['collected']:
                hit_x = DINO_X < coin['x'] + COIN_SIZE and DINO_X + DINO_W > coin['x']
                hit_y = self.dino_y < coin['y'] + COIN_SIZE and self.dino_y + DINO_H > coin['y']
                if hit_x and hit_y:
                    # Case 3: coin collected
                    coin['collected'] = True
                    self.coins_collected += 1
                    reward += REWARD_COIN_COLLECT
        self.coins = [co for co in self.coins
                      if co['x'] + COIN_SIZE > -OFFSCREEN_MARGIN and not (co['collected'] and co['x'] + COIN_SIZE < DINO_X)]

        self._coin_dist += CACTUS_SPEED
        if self._coin_dist >= self._coin_gap:
            self._coin_dist = 0.0
            self._coin_gap  = random.randint(COIN_GAP_MIN, COIN_GAP_MAX)
            self.coins.append({'x': CANVAS_W + SPAWN_MARGIN, 'y': rand_coin_y(), 'collected': False})

        for c in self.cacti:
            hit_x = DINO_X < c['x'] + c['w'] and DINO_X + DINO_W > c['x']
            hit_y = self.dino_y + DINO_H > GROUND_Y - c['h']
            if hit_x and hit_y:
                # Case 4: crash penalty
                reward += REWARD_CRASH
                done = True
                break

        if self.step_count >= self.max_steps:
            done = True

        return self._get_state(), reward, done, {}


# ======================================================================
# 3. PURE GREEDY TABULAR Q-LEARNING AGENT
# ======================================================================

class QLearningAgent:
    def __init__(self):
        self.q_table = np.full((NUM_STATES, NUM_ACTIONS), INIT_Q)

    def select_action(self, state, epsilon=0.0):
        if random.random() < epsilon:
            return random.randrange(NUM_ACTIONS)
        q = self.q_table[state]
        best = np.where(q == q.max())[0]
        return int(random.choice(best))

    def update(self, s, a, r, s_next, done):
        target = r if done else r + GAMMA * self.q_table[s_next].max()
        self.q_table[s, a] += ALPHA * (target - self.q_table[s, a])


# ======================================================================
# 4. TRAINING & EVALUATION
# ======================================================================

def train_dino_agent():
    random.seed(SEED)
    np.random.seed(SEED)
    env   = ChromeDinoEnv()
    agent = QLearningAgent()

    steps_hist = []
    coins_hist = []

    print("=" * 72)
    print(f"  CHROME DINO TABULAR Q-LEARNING: {NUM_ACTIONS} ACTIONS, JUMP-PHYSICS STATE SPACE")
    print("=" * 72)
    print(f"States: {NUM_STATES} | Actions: {NUM_ACTIONS} | Episodes: {TRAIN_EPISODES}")
    print(f"Alpha: {ALPHA} | Gamma: {GAMMA} | Epsilon: {EPSILON_START} -> {EPSILON_END} | Init Q: {INIT_Q}")
    print(f"Coin gap: {COIN_GAP_MIN}-{COIN_GAP_MAX} | Max steps: {MAX_STEPS}")
    print("-" * 72)
    print(f"{'Episode':^9} | {'Steps':^8} | {'Cactuses':^9} | {'Coins':^7} | {'Reward':^9}")
    print("-" * 72)

    for ep in range(1, TRAIN_EPISODES + 1):
        frac       = (ep - 1) / max(1, TRAIN_EPISODES - 1)
        epsilon    = EPSILON_START + (EPSILON_END - EPSILON_START) * frac
        state      = env.reset()
        ep_reward = 0.0
        done      = False

        while not done:
            action               = agent.select_action(state, epsilon)
            next_state, r, done, _ = env.step(action)
            agent.update(state, action, r, next_state, done)
            state                = next_state
            ep_reward           += r

        steps_hist.append(env.step_count)
        coins_hist.append(env.coins_collected)

        if ep == 1 or ep % LOG_EVERY == 0:
            print(f"{ep:^9d} | {env.step_count:^8d} | {env.cacti_cleared:^7d} | "
                  f"{env.coins_collected:^7d} | {ep_reward:^9.1f}")

    print("-" * 72)
    print("Training complete.\n" + "=" * 72)
    return env, agent, steps_hist, coins_hist


def evaluate_agent(env, agent):
    print("=" * 72)
    print(f"  GREEDY EVALUATION ({EVAL_STEPS:,} STEPS)")
    print("=" * 72)
    print(f"{'No.':^9} | {'Steps':^12} | {'Cactuses':^9} | {'Coins':^7} | {'Status':^14}")
    print("-" * 72)

    for ep in range(1, 6):
        state = env.reset()
        done  = False
        while not done:
            action             = agent.select_action(state)
            state, _, done, _  = env.step(action)
        status = "PASSED" if env.step_count >= EVAL_STEPS else "CRASHED"
        print(f"{ep:^9d} | {env.step_count:^12,d} | {env.cacti_cleared:^9,d} | "
              f"{env.coins_collected:^7,d} | {status:^14}")

    print("=" * 72 + "\n")


# ======================================================================
# 5. TRAINING CURVE PLOT
# ======================================================================

def plot_training(agent, steps_hist, coins_hist,
                  output_path="section/ch07_figures/fig_7_02_dino_learning_and_policy.png"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.0, 5.0), dpi=300)
    plt.subplots_adjust(wspace=0.32, left=0.07, right=0.96, top=0.88, bottom=0.14)

    ep_axis = np.arange(1, len(steps_hist) + 1)

    ax1.plot(ep_axis, steps_hist, color='#1E40AF', linewidth=1.1, alpha=0.9, label='Steps survived')
    ax1.set_xlabel('Training Episode', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Steps Survived', fontsize=11, fontweight='bold', color='#1E40AF')
    ax1.tick_params(axis='y', labelcolor='#1E40AF')
    ax1.set_ylim(0, MAX_STEPS + 100)
    ax1.grid(True, linestyle='--', alpha=0.4)
    max_steps = max(steps_hist) if steps_hist else 0
    ax1.axhline(max_steps, color='#1E40AF', linewidth=0.8, linestyle=':', alpha=0.6)
    ax1.annotate(f'best: {max_steps:,} steps', xy=(ep_axis[-1], max_steps),
                 xytext=(-8, -14), textcoords='offset points',
                 ha='right', color='#1E40AF', fontsize=9, fontweight='bold')

    ax1b = ax1.twinx()
    ax1b.plot(ep_axis, coins_hist, color='#D97706', linewidth=1.1, alpha=0.9, linestyle='--', label='Coins collected')
    ax1b.set_ylabel('Coins per Episode', fontsize=11, fontweight='bold', color='#D97706')
    ax1b.tick_params(axis='y', labelcolor='#D97706')
    max_coins = max(coins_hist) if coins_hist else 0
    ax1b.set_ylim(-0.5, max(10, max_coins * 1.15))

    lines  = [ax1.lines[0], ax1b.lines[0]]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', fontsize=9, framealpha=0.9)
    ax1.set_title('(a) Q-Learning Training Curves', fontsize=12, fontweight='bold', pad=10)

    categories = [f"{b*OBS_BIN_W}-{b*OBS_BIN_W + OBS_BIN_W - 1}" for b in range(NUM_DIST_BINS)]
    states = [b * NUM_AIRBORNE for b in range(NUM_DIST_BINS)]  # obs_type=0, grounded

    x   = np.arange(len(categories))
    w   = 0.26
    ax2.bar(x - w, [agent.q_table[s, 0] for s in states], w, label='Q(RUN)',    color='#0284C7')
    ax2.bar(x,     [agent.q_table[s, 1] for s in states], w, label='Q(SHORT)', color='#10B981')
    ax2.bar(x + w, [agent.q_table[s, 2] for s in states], w, label='Q(LONG)',  color='#EF4444')

    ax2.axhline(0, color='#334155', linewidth=1.0, linestyle='-', alpha=0.5)
    q_vals = np.concatenate([agent.q_table[s] for s in states])
    q_min, q_max = q_vals.min(), q_vals.max()
    pad = max(1.0, (q_max - q_min) * 0.15)
    ax2.set_ylim(q_min - pad, q_max + pad)

    ax2.set_xlabel('Obstacle Distance (px)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Q(s, a)', fontsize=11, fontweight='bold')
    ax2.set_title('(b) Learned Action Values by Obstacle Distance', fontsize=12, fontweight='bold', pad=10)
    ax2.set_xticks(x)
    ax2.set_xticklabels(categories, fontsize=9)
    for tick in ax2.get_yticks():
        ax2.axhline(tick, color='#94A3B8', linewidth=0.5, linestyle='--', alpha=0.25, zorder=0)
    ax2.grid(False)
    ax2.legend(fontsize=9, loc='upper left', framealpha=0.9)
    ax2.grid(True, linestyle='--', alpha=0.3, axis='y')

    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure to: {output_path}")


# ======================================================================
# 6. MAIN
# ======================================================================

if __name__ == "__main__":
    env, agent, steps_hist, coins_hist = train_dino_agent()
    evaluate_agent(ChromeDinoEnv(max_steps=EVAL_STEPS), agent)
    plot_training(agent, steps_hist, coins_hist,
                  "section/ch07_figures/fig_7_02_dino_learning_and_policy.png")
    render_colab_dino(agent)
